# Practical Task 7: Batch Prediction Pipeline

This notebook implements a **batch prediction pipeline** that:
1. Creates a SQLite database with `input_data` and `predictions` tables
2. Loads a trained ML model (`model.joblib`)
3. Runs batch predictions on all input data
4. Stores results back into the database with timestamps
5. Executes automatically on a schedule (every 5 minutes)

**Dataset:** Iris flower species classification  
**Model:** Logistic Regression (trained in Practice 6)  
**Database:** SQLite (built into Python, no installation needed)

## Section 1: Setup & Imports

In [ ]:
# Install required library if not already installed
import subprocess
subprocess.run(['pip', 'install', 'schedule'], capture_output=True)

import sqlite3
import joblib
import numpy as np
import pandas as pd
import schedule
import time
import threading
from datetime import datetime

print('All imports successful!')
print(f'Run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Section 2: Database Setup

We use **SQLite** — it stores everything in a single `.db` file and requires zero installation.

Two tables:
- `input_data` — flowers waiting to be classified (id + 4 measurements)
- `predictions` — results written by our pipeline (id + prediction + timestamp)

In [ ]:
# Database file path
DB_PATH = 'iris_pipeline.db'

def create_database():
    """Create the database and both required tables."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Table 1: input_data — stores flowers to predict
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS input_data (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            sepal_length    REAL NOT NULL,
            sepal_width     REAL NOT NULL,
            petal_length    REAL NOT NULL,
            petal_width     REAL NOT NULL
        )
    ''')

    # Table 2: predictions — stores results from the pipeline
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS predictions (
            id                   INTEGER PRIMARY KEY,
            prediction           TEXT NOT NULL,
            prediction_timestamp TEXT NOT NULL
        )
    ''')

    conn.commit()
    conn.close()
    print('Database created: iris_pipeline.db')
    print('Tables created: input_data, predictions')

create_database()

## Section 3: Insert Sample Input Data

In a real system, this data would come from sensors, user uploads, or another service.
Here we insert **30 sample iris flowers** to simulate a real batch workload.

In [ ]:
# Sample iris flower measurements
# (sepal_length, sepal_width, petal_length, petal_width)
sample_flowers = [
    # Setosa samples
    (5.1, 3.5, 1.4, 0.2),
    (4.9, 3.0, 1.4, 0.2),
    (4.7, 3.2, 1.3, 0.2),
    (4.6, 3.1, 1.5, 0.2),
    (5.0, 3.6, 1.4, 0.2),
    (5.4, 3.9, 1.7, 0.4),
    (4.6, 3.4, 1.4, 0.3),
    (5.0, 3.4, 1.5, 0.2),
    (4.4, 2.9, 1.4, 0.2),
    (4.9, 3.1, 1.5, 0.1),
    # Versicolor samples
    (7.0, 3.2, 4.7, 1.4),
    (6.4, 3.2, 4.5, 1.5),
    (6.9, 3.1, 4.9, 1.5),
    (5.5, 2.3, 4.0, 1.3),
    (6.5, 2.8, 4.6, 1.5),
    (5.7, 2.8, 4.5, 1.3),
    (6.3, 3.3, 4.7, 1.6),
    (4.9, 2.4, 3.3, 1.0),
    (6.6, 2.9, 4.6, 1.3),
    (5.2, 2.7, 3.9, 1.4),
    # Virginica samples
    (6.3, 3.3, 6.0, 2.5),
    (5.8, 2.7, 5.1, 1.9),
    (7.1, 3.0, 5.9, 2.1),
    (6.3, 2.9, 5.6, 1.8),
    (6.5, 3.0, 5.8, 2.2),
    (7.6, 3.0, 6.6, 2.1),
    (4.9, 2.5, 4.5, 1.7),
    (7.3, 2.9, 6.3, 1.8),
    (6.7, 2.5, 5.8, 1.8),
    (7.2, 3.6, 6.1, 2.5),
]

def insert_sample_data():
    """Insert sample flowers into input_data table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Only insert if table is empty (avoid duplicates on re-run)
    cursor.execute('SELECT COUNT(*) FROM input_data')
    count = cursor.fetchone()[0]

    if count == 0:
        cursor.executemany(
            'INSERT INTO input_data (sepal_length, sepal_width, petal_length, petal_width) VALUES (?, ?, ?, ?)',
            sample_flowers
        )
        conn.commit()
        print(f'Inserted {len(sample_flowers)} sample flowers into input_data')
    else:
        print(f'input_data already has {count} rows — skipping insert')

    conn.close()

insert_sample_data()

# Preview the input data
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql('SELECT * FROM input_data', conn)
conn.close()
print(f'\nInput data preview ({len(df_input)} rows):')
df_input.head(10)

## Section 4: Load the Trained Model

We reuse the **same model.joblib** from Practice 6 / SIS-3.  
No need to retrain — training was done once and saved.

In [ ]:
# Species name mapping
SPECIES_NAMES = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}

def load_model(model_path='model.joblib'):
    """Load the trained model from disk."""
    model = joblib.load(model_path)
    print(f'Model loaded: {type(model).__name__}')
    print(f'Classes: {[SPECIES_NAMES[i] for i in model.classes_]}')
    print(f'Number of features: {model.n_features_in_}')
    return model

model = load_model()

## Section 5: Batch Prediction Function

This is the **core of the pipeline**. It:
1. Connects to the database
2. Reads all rows from `input_data` that haven't been predicted yet
3. Runs the model on ALL of them at once (this is what makes it "batch")
4. Writes results to `predictions` table with a timestamp

In [ ]:
def run_batch_prediction():
    """
    Core batch prediction function.
    Reads unprocessed input data, predicts species, stores results.
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'\n{"="*50}')
    print(f'Batch prediction run started: {timestamp}')
    print(f'{"="*50}')

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Read only rows that haven't been predicted yet
    # (input_data rows with no matching id in predictions)
    query = '''
        SELECT i.id, i.sepal_length, i.sepal_width, i.petal_length, i.petal_width
        FROM input_data i
        LEFT JOIN predictions p ON i.id = p.id
        WHERE p.id IS NULL
    '''
    df = pd.read_sql(query, conn)

    if df.empty:
        print('No new data to predict. Pipeline is up to date.')
        conn.close()
        return

    print(f'Found {len(df)} new records to predict')

    # Extract features as numpy array — this is the BATCH
    features = df[['sepal_length', 'sepal_width',
                   'petal_length', 'petal_width']].values

    # Run model on ALL records at once (batch prediction)
    predictions = model.predict(features)
    species_names = [SPECIES_NAMES[p] for p in predictions]

    # Write results to predictions table
    results = [
        (int(row_id), species, timestamp)
        for row_id, species in zip(df['id'], species_names)
    ]

    cursor.executemany(
        'INSERT INTO predictions (id, prediction, prediction_timestamp) VALUES (?, ?, ?)',
        results
    )
    conn.commit()

    # Show summary
    print(f'Successfully predicted {len(results)} records')
    print(f'Results saved to predictions table at {timestamp}')

    # Show prediction distribution
    from collections import Counter
    dist = Counter(species_names)
    print(f'\nPrediction breakdown:')
    for species, count in dist.items():
        print(f'  {species}: {count} flowers')

    conn.close()

print('Batch prediction function defined successfully!')

## Section 6: Run the First Batch

Let's run the pipeline manually once to see it working.

In [ ]:
# Run batch prediction manually — first time
run_batch_prediction()

## Section 7: View Results in the Database

In [ ]:
def show_results():
    """Display the full predictions table joined with input data."""
    conn = sqlite3.connect(DB_PATH)

    query = '''
        SELECT
            i.id,
            i.sepal_length,
            i.sepal_width,
            i.petal_length,
            i.petal_width,
            p.prediction,
            p.prediction_timestamp
        FROM input_data i
        JOIN predictions p ON i.id = p.id
        ORDER BY i.id
    '''

    df = pd.read_sql(query, conn)
    conn.close()

    print(f'Total predictions stored: {len(df)}')
    print()
    return df

results_df = show_results()
results_df

## Section 8: Add New Data & Run Again

This simulates what happens in production — new data arrives, the pipeline picks it up in the next scheduled run.

In [ ]:
def add_new_flowers(flowers):
    """Simulate new data arriving — insert new rows into input_data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.executemany(
        'INSERT INTO input_data (sepal_length, sepal_width, petal_length, petal_width) VALUES (?, ?, ?, ?)',
        flowers
    )
    conn.commit()
    conn.close()
    print(f'Added {len(flowers)} new flowers to input_data')

# Simulate new flowers arriving
new_batch = [
    (5.2, 3.4, 1.4, 0.2),   # should be setosa
    (6.1, 2.8, 4.7, 1.2),   # should be versicolor
    (7.2, 3.0, 5.8, 1.6),   # should be virginica
    (6.0, 2.7, 4.5, 1.4),   # borderline versicolor/virginica
    (5.0, 3.5, 1.3, 0.3),   # should be setosa
]

add_new_flowers(new_batch)
print('\nRunning pipeline on new data...')
run_batch_prediction()

In [ ]:
# Show updated results — notice the new rows have a different timestamp
results_df = show_results()
results_df.tail(10)

## Section 9: Scheduler

The pipeline runs automatically every **5 minutes** using the `schedule` library.

It runs in a **background thread** so the notebook stays interactive.

> In production, you would use **cron** (Linux) or a task scheduler instead of running it inside a notebook.

In [ ]:
# Track whether scheduler is running
scheduler_running = False
scheduler_thread = None

def scheduler_loop():
    """Background loop that runs the scheduler."""
    global scheduler_running
    print('Scheduler started — pipeline will run every 5 minutes')
    print('(Running once immediately, then every 5 minutes)')
    while scheduler_running:
        schedule.run_pending()
        time.sleep(10)  # check every 10 seconds
    print('Scheduler stopped.')

def start_scheduler():
    """Start the batch prediction scheduler."""
    global scheduler_running, scheduler_thread

    if scheduler_running:
        print('Scheduler is already running!')
        return

    # Clear any existing jobs
    schedule.clear()

    # Schedule the pipeline every 5 minutes
    schedule.every(5).minutes.do(run_batch_prediction)

    # Run once immediately so you can see it working right away
    run_batch_prediction()

    # Start background thread
    scheduler_running = True
    scheduler_thread = threading.Thread(target=scheduler_loop, daemon=True)
    scheduler_thread.start()

def stop_scheduler():
    """Stop the scheduler."""
    global scheduler_running
    scheduler_running = False
    schedule.clear()
    print('Scheduler stopped.')

print('Scheduler functions defined!')
print('Run start_scheduler() to begin automatic execution')
print('Run stop_scheduler() to stop it')

In [ ]:
# Start the automatic scheduler
# It will run the batch pipeline immediately and then every 5 minutes
start_scheduler()

In [ ]:
# Wait 30 seconds and add more new data
# This simulates new data arriving between scheduled runs
print('Adding more new flowers to simulate incoming data...')

more_flowers = [
    (4.8, 3.1, 1.6, 0.2),
    (6.7, 3.0, 5.0, 1.7),
    (6.2, 2.9, 4.3, 1.3),
]

add_new_flowers(more_flowers)
print('These will be picked up in the next scheduled run (within 5 minutes)')
print('Or you can run manually:')
print('  run_batch_prediction()')

In [ ]:
# Run manually to process the new flowers immediately
run_batch_prediction()

## Section 10: Final Summary — All Predictions

In [ ]:
# Show complete final state of the database
conn = sqlite3.connect(DB_PATH)

# Summary statistics
total_input = pd.read_sql('SELECT COUNT(*) as count FROM input_data', conn).iloc[0]['count']
total_pred  = pd.read_sql('SELECT COUNT(*) as count FROM predictions', conn).iloc[0]['count']

print('='*50)
print('PIPELINE SUMMARY')
print('='*50)
print(f'Total flowers in input_data:  {int(total_input)}')
print(f'Total predictions stored:     {int(total_pred)}')
print(f'Unprocessed:                  {int(total_input - total_pred)}')
print()

# Prediction distribution
dist = pd.read_sql(
    'SELECT prediction, COUNT(*) as count FROM predictions GROUP BY prediction',
    conn
)
print('Prediction distribution:')
print(dist.to_string(index=False))
print()

# Runs by timestamp (each unique timestamp = one batch run)
runs = pd.read_sql(
    'SELECT prediction_timestamp, COUNT(*) as records_processed FROM predictions GROUP BY prediction_timestamp ORDER BY prediction_timestamp',
    conn
)
print('Batch runs history:')
print(runs.to_string(index=False))

conn.close()

In [ ]:
# Show full predictions table
final_df = show_results()
final_df

In [ ]:
# Stop the scheduler when done
stop_scheduler()
print('Pipeline demonstration complete!')

## Summary

This notebook implemented a complete **batch prediction pipeline**:

| Component | Implementation |
|---|---|
| **Database** | SQLite (`iris_pipeline.db`) |
| **Input table** | `input_data` (id, sepal_length, sepal_width, petal_length, petal_width) |
| **Output table** | `predictions` (id, prediction, prediction_timestamp) |
| **Model** | Logistic Regression loaded from `model.joblib` |
| **Batch logic** | Reads only unprocessed rows, predicts all at once |
| **Scheduler** | Runs automatically every 5 minutes via `schedule` library |

### Key difference from real-time prediction (SIS-3)

| | SIS-3 (Real-time) | Practical Task 7 (Batch) |
|---|---|---|
| **Trigger** | User clicks Predict | Automatic timer |
| **Records** | 1 at a time | Many at once |
| **Speed** | Instant response needed | Can take minutes |
| **Storage** | Returns JSON response | Writes to database |
| **Use case** | Interactive UI | Overnight processing |